<img src="https://raw.githubusercontent.com/brazil-data-cube/rstac/master/inst/extdata/img/logo.png" align="right" width="110" />
<img src="https://brazil-data-cube.github.io/_images/logo-bdc.png" align="right" width="130" />

#  Introdução ao pacote rstac 

**R Client Library for SpatioTemporal Asset Catalog (rstac)**

Rolf Simões, Felipe Carvalho, Gilberto Ribeiro, Karine Reis

Disponível em: [github.com/brazil-data-cube/rstac](https://github.com/brazil-data-cube/rstac)

<hr style="border: 1px solid #0984e3;">

## Sumário

- [**Sobre o `rstac`**](#about-0)
- [**1. Primeiros passos**](#sec-1-1)
    - [**1.1 Importação de pacotes**](#sec-1-2)
- [**2. Realizando consultas**](#sec-2)
- [**3. Funções auxiliares**](#sec-5)
- [**4. Baixando imagens**](#sec-6)
- [**5. Referências**](#sec-8)



<hr style="border: 1px solid #0984e3;">

## Sobre o `rstac` <a id="about-0"></a>

<img src="https://raw.githubusercontent.com/OldLipe/exemplo_rstac/master/img/rstac_diagram.png" align="right" width="700" />
<br>

<p style="text-align:left">
O pacote rstac foi concebido para suportar totalmente as versões de 0.8.1 até as mais recentes do STAC.

Além de suportar os endpoints fornecidos pela especificação, o pacote rstac fornece diversas funções auxiliares.  
    
<p style="text-align:left">

<hr style="border: 1px solid #0984e3;">

## 1. Primeiros passos  <a id="sec-1"></a>

### 1.1 Importação dos pacotes <a id="sec-1-2"></a>

In [ ]:
library(rstac)    # Pacote rstac
library(terra)    # Pacote para manipular e visualizar imagens

##  2. Realizando consultas <a id="sec-2"></a>

<img src="https://raw.githubusercontent.com/OldLipe/exemplo_rstac/master/img/diagram_01_01.png" align="right" width="500" />

Nesta seção, vamos criar algumas consultas usando as funções do pacote `rstac`.

In [ ]:
s_obj <- stac("https://data.inpe.br/bdc/stac/v1/")

s_obj |> get_request() |> print()

In [ ]:
s_obj |> collections() |> get_request() |> print()

In [ ]:
s_obj |> collections("CBERS-WFI-8D-1") |> get_request() |> print()

### 2.1 Buscando imagens com filtros

In [ ]:
r_get <- s_obj |> 
    stac_search(collections = "CBERS-WFI-8D-1", 
                limit = 100) |> 
    post_request()

print(r_get)

In [ ]:
r_get <- s_obj |> 
    stac_search(collections = "CBERS-WFI-8D-1", 
                datetime = "2023-07-01/2023-09-29", 
                limit = 100) |> 
    post_request()

print(r_get)

In [ ]:
r_get <- s_obj |> 
    stac_search(collections = "CBERS-WFI-8D-1", 
                datetime = "2023-07-01/2023-09-29", 
                bbox = c(-45.8976,-23.2191,-45.8901,-23.2138),
                limit = 100) |> 
    post_request()

print(r_get)

In [ ]:
r_get <- s_obj |> 
    stac_search(collections = "CBERS-WFI-8D-1", 
                datetime = "2023-07-01/2023-09-29", 
                limit = 100) |> 
    ext_query("bdc:tiles" == "007007") |>
    post_request()

print(r_get)

In [ ]:
rstac::items_reap(r_get, c("properties", "eo:cloud_cover"))

In [ ]:
r_get <- s_obj |> 
    stac_search(collections = "CBERS-WFI-8D-1", 
                datetime = "2023-07-01/2023-09-29", 
                limit = 10) |> 
    ext_query("bdc:tiles" == "007007", 
              'eo:cloud_cover' < 3) |>
    post_request()

print(r_get)

In [ ]:
rstac::items_reap(r_get, c("properties", "eo:cloud_cover"))

## 3. Funções auxiliares  <a id="sec-5"></a>

Os objetos do tipo `stac_item_collection` possuem algumas funções facilitadoras para manipular/extrair informações deste objeto, sendo elas:


- **`item_fetch()`:** Realiza a paginação dos assets
- **`item_length()`:** Retorna a quantidade de itens em um objeto
- **`item_matched()`:** Retorna a quantidade de itens corresponderam aos critérios de pesquisa

#### 3.1 `items_fetch()`

In [ ]:
items_f <- s_obj |> 
    stac_search(collections = "CBERS-WFI-8D-1", 
                datetime = "2023-01-01/2023-12-31", 
                limit = 50) |> 
    ext_query("bdc:tiles" %in% c("007007", "006007")) |>
    post_request()

In [ ]:
print(items_f)

In [ ]:
items_new <- items_f |> items_fetch()

print(items_new)

#### 3.2 `items_matched()`

In [ ]:
items_matched(items_f)

#### 3.3 `item_length()`

In [ ]:
items_length(items_new)

## 4. Baixando imagens <a id="sec-6"></a>

Além das funções mencionadas acima, é possível realizar o *dowload* de todos os *assets* retornados de uma pesquisa. Para isso, pode ser usado o método: `assets_download()`. Como apresentado na célula abaixo.

In [ ]:
r_get <- s_obj |> 
    stac_search(collections = "CBERS-WFI-8D-1", 
                datetime = "2023-07-01/2023-09-29", 
                limit = 1) |> 
    ext_query("bdc:tiles" == "007007", 
              'eo:cloud_cover' < 3) |>
    post_request() |>
    assets_download(asset_names = c("BAND13", "BAND14", "BAND15"), overwrite = TRUE, output_dir = ".")

print(r_get)

In [ ]:
r <- terra::rast(rstac::assets_url(r_get))

In [ ]:
r

In [ ]:
plotRGB(r, r = 3, g = 2, b = 1, stretch = "hist", scale = 10000)

## 5. Referências <a id="sec-8"></a>

- [1] https://stacspec.org/
- [2] https://github.com/brazil-data-cube/stac.py